### 서울/경기 도시데이터 센서(S-DoT) 환경정보

In [2]:
%use dataframe
%use datetime
%use ktor-client

##### 서울시 도시데이터 센서(S-DoT) 환경정보 설치 위치정보
##### 출처:
https://data.seoul.go.kr/dataList/OA-15969/S/1/datasetView.do

In [3]:
val url = "./data/서울시 도시데이터 센서(S-DoT) 환경정보 설치 위치정보.xlsx"
val df_observatory = DataFrame.readExcel(url).update { colsOf<String>() }.with { it.trim() }
df_observatory.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
No,Double,1170,1170,0,1.000000,1,585.500000,337.894214,1.000000,292.916667,585.500000,878.083333,1170.000000
모델 시리얼(*),String,1170,1170,0,V02Q1940655,1,null,null,OC3CL200010,OC3CL240001,V02Q1940329,V02Q1940624,V02Q2300007
주소,String,1170,1155,0,서울대공원,4,null,null,서울 강동구 강일동 685,서울특별시 강서구 등촌로51가길 26,서울특별시 동작구 상도로 181,서울특별시 송파구 중대로4길 8,서울특별시 중랑구 중랑천로24길 19
좌표 구분코드,String,1170,1,0,W84,1170,null,null,W84,W84,W84,W84,W84
위도,Comparable<*>,1170,1167,0,37.568686,2,null,null,null,null,null,null,null
경도,Comparable<*>,1170,1166,0,127.080180,2,null,null,null,null,null,null,null
변경 전 시리얼,String?,1170,28,1143,OC3CL2000086,1,null,null,OC3CL2000086,OC3DL2200002,OC3DL2200010,OC3DL2200016,OC3KL2400125
변경 전 시리얼(데이터 상 표기),String?,1170,6,1143,OC3DL220001,10,null,null,OC3CL200008,OC3DL220000,OC3DL220001,OC3DL220001,OC3KL240012


~~~
USE { dependencies("org.xerial:sqlite-jdbc:3.51.1.0") }

import java.sql.DriverManager

val dbPath = "jdbc:sqlite:/Users/unchil/AndroidStudioProjects/OceanWaterInfo/oceanwater.sqlite"
val tableName = "SDoT_Location"
DriverManager.getConnection(dbPath).use{ conn ->
    val sql = """INSERT INTO ${tableName} (serial, addr, lat, lng) VALUES (?,?,?,? )""".trimIndent()
    df.select("모델 시리얼(*)", "주소", "위도", "경도").forEach { it ->
        try {
            conn.prepareStatement(sql)?.use { preparedStatement ->
                preparedStatement.setString(1, it["모델 시리얼(*)"].toString())
                preparedStatement.setString(2, it["주소"].toString())
                preparedStatement.setString(3, it["위도"].toString())
                preparedStatement.setString(4, it["경도"].toString())
                preparedStatement.executeUpdate()
            }
        } catch (e: Exception){
            println(e.localizedMessage)
        }
    }

}
~~~

In [4]:
import kotlin.random.Random

val df_ObservatoryRename = df_observatory.select("모델 시리얼(*)", "주소", "위도", "경도").rename(
    "모델 시리얼(*)" to "SERIAL",
    "주소" to "addr",
    "위도" to "lat",
    "경도" to "lng"
)

df_ObservatoryRename.describe()

name,type,count,unique,nulls,top,freq,min,p25,median,p75,max
SERIAL,String,1170,1170,0,V02Q1940655,1,OC3CL200010,OC3CL240001,V02Q1940329,V02Q1940624,V02Q2300007
addr,String,1170,1155,0,서울대공원,4,서울 강동구 강일동 685,서울특별시 강서구 등촌로51가길 26,서울특별시 동작구 상도로 181,서울특별시 송파구 중대로4길 8,서울특별시 중랑구 중랑천로24길 19
lat,Comparable<*>,1170,1167,0,37.568686,2,null,null,null,null,null
lng,Comparable<*>,1170,1166,0,127.080180,2,null,null,null,null,null


1. 현재 S-DoT 장비의 unique count 값이 1170.
2. 최초 1000 건을 수집하되 SENSING_TIME 이 unique 하면 200 건을 더 수집.
3. 수집된 데이터중 SENSING_TIME 이 MAX(SENSING_TIME) 인 값만 filtering.

In [5]:
val apiKey = ""
val start_index = 1
val end_index = 1000
val url = "http://openapi.seoul.go.kr:8088/${apiKey}/json/sDoTEnv/${start_index}/${end_index}"

val df_ObservationRaw = DataFrame.readJson(url).update { colsOf<String>() }.with { it.trim() }
df_ObservationRaw.describe()

name,path,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
list_total_count,"[sDoTEnv, list_total_count]",Int,1,1,0,437808,1,437808.000000,NaN,437808,437808.000000,437808.000000,437808.000000,437808
CODE,"[sDoTEnv, RESULT, CODE]",String,1,1,0,INFO-000,1,null,null,INFO-000,INFO-000,INFO-000,INFO-000,INFO-000
MESSAGE,"[sDoTEnv, RESULT, MESSAGE]",String,1,1,0,정상 처리되었습니다,1,null,null,정상 처리되었습니다,정상 처리되었습니다,정상 처리되었습니다,정상 처리되었습니다,정상 처리되었습니다
MODELNAME,"[sDoTEnv, row, MODELNAME]",String,1000,1,0,SDOT001,1000,null,null,SDOT001,SDOT001,SDOT001,SDOT001,SDOT001
SERIAL,"[sDoTEnv, row, SERIAL]",String,1000,980,0,V02Q1941015,2,null,null,OC3CL200011,OC3KL240012,V02Q1940350,V02Q1940659,V02Q2300007
SENSING_TIME,"[sDoTEnv, row, SENSING_TIME]",String,1000,2,0,2026-06-10_13:07:00,980,null,null,2026-06-10_12:07:00,2026-06-10_13:07:00,2026-06-10_13:07:00,2026-06-10_13:07:00,2026-06-10_13:07:00
REGION,"[sDoTEnv, row, REGION]",String,1000,8,0,residential_area,704,null,null,commercial_area,residential_area,residential_area,residential_area,traditional_markets
AUTONOMOUS_DISTRICT,"[sDoTEnv, row, AUTONOMOUS_DISTRICT]",String,1000,26,0,Yangcheon-gu,77,null,null,Dobong-gu,Gangnam-gu,Jongno-gu,Seongbuk-gu,Yongsan-gu
ADMINISTRATIVE_DISTRICT,"[sDoTEnv, row, ADMINISTRATIVE_DISTRICT]",String,1000,402,0,Amsa1(il)-dong,11,null,null,Ahyeon-dong,Gangil-dong,Junghwa2(i)-dong,Seongsan1(il)-dong,valet_parking1
MAX_TEMP,"[sDoTEnv, row, MAX_TEMP]",String,1000,67,0,,66,null,null,,26.4,27.2,27.9,31.3


In [6]:
val currentSensingTime = df_ObservationRaw.sDoTEnv.row[0].SENSING_TIME.max()
currentSensingTime

2026-06-10_13:07:00

In [7]:
val df_FilteredObservationValues = df_ObservationRaw.sDoTEnv.row[0].filter { it.SENSING_TIME == currentSensingTime }
df_FilteredObservationValues.describe()

name,type,count,unique,nulls,top,freq,min,p25,median,p75,max
MODELNAME,String,980,1,0,SDOT001,980,SDOT001,SDOT001,SDOT001,SDOT001,SDOT001
SERIAL,String,980,980,0,V02Q1940706,1,OC3CL200011,OC3DL220021,V02Q1940340,V02Q1940643,V02Q2300007
SENSING_TIME,String,980,1,0,2026-06-10_13:07:00,980,2026-06-10_13:07:00,2026-06-10_13:07:00,2026-06-10_13:07:00,2026-06-10_13:07:00,2026-06-10_13:07:00
REGION,String,980,8,0,residential_area,697,commercial_area,residential_area,residential_area,residential_area,traditional_markets
AUTONOMOUS_DISTRICT,String,980,26,0,Yangcheon-gu,73,Dobong-gu,Gangnam-gu,Jongno-gu,Seongbuk-gu,Yongsan-gu
ADMINISTRATIVE_DISTRICT,String,980,402,0,Amsa1(il)-dong,11,Ahyeon-dong,Gangil-dong,Junghwa1(il)-dong,Seongnae3(sam)-dong,valet_parking1
MAX_TEMP,String,980,67,0,,64,,26.4,27.2,27.9,31.3
AVG_TEMP,String,980,62,0,,64,,25.7,26.5,27.2,30.4
MIN_TEMP,String,980,62,0,,64,,25.0,25.8,26.5,29.8
MAX_HUMI,String,980,43,0,,155,,51.,55.,58.,89.


In [8]:
val df_Joined = df_ObservatoryRename.innerJoin(df_FilteredObservationValues) { SERIAL }
val df_Final = df_Joined.select(
    "SENSING_TIME", "SERIAL", "MODELNAME", "addr", "AUTONOMOUS_DISTRICT",
    "ADMINISTRATIVE_DISTRICT", "lat", "lng", "MAX_TEMP", "MAX_HUMI",
    "MAX_WIND_SPEED", "MAX_WIND_DIRE", "MAX_INTE_ILLU", "MAX_ULTRA_RAYS",
    "MAX_NOISE", "MAX_VIBR_X", "MAX_VIBR_Y", "MAX_VIBR_Z", "MAX_EFFE_TEMP",
    "MAX_NO2", "MAX_CO", "MAX_SO2", "MAX_NH3", "MAX_H2S", "MAX_O3"
).rename { all() }.into { it.name.removePrefix("MAX_").lowercase() }

df_Final.describe()

name,type,count,unique,nulls,top,freq,min,p25,median,p75,max
sensing_time,String,980,1,0,2026-06-10_13:07:00,980,2026-06-10_13:07:00,2026-06-10_13:07:00,2026-06-10_13:07:00,2026-06-10_13:07:00,2026-06-10_13:07:00
serial,String,980,980,0,V02Q1940655,1,OC3CL200011,OC3DL220021,V02Q1940340,V02Q1940643,V02Q2300007
modelname,String,980,1,0,SDOT001,980,SDOT001,SDOT001,SDOT001,SDOT001,SDOT001
addr,String,980,970,0,서울특별시 송파구 신천동 32,3,서울 강동구 강일동 685,서울특별시 강서구 마곡동 749-2,서울특별시 동작구 상도로37길 48,서울특별시 송파구 위례성대로2길 34,서울특별시 중랑구 중랑천로24길 19
autonomous_district,String,980,26,0,Yangcheon-gu,73,Dobong-gu,Gangnam-gu,Jongno-gu,Seongbuk-gu,Yongsan-gu
administrative_district,String,980,402,0,Amsa1(il)-dong,11,Ahyeon-dong,Gangil-dong,Junghwa1(il)-dong,Seongnae3(sam)-dong,valet_parking1
lat,Comparable<*>,980,978,0,37.471295,2,null,null,null,null,null
lng,Comparable<*>,980,977,0,126.976553,2,null,null,null,null,null
temp,String,980,67,0,,64,,26.4,27.2,27.9,31.3
humi,String,980,43,0,,155,,51.,55.,58.,89.


##### 경기도 도시데이터 센서(S-DoT) 환경정보 설치 위치정보
##### 출처:
###### 경기도의 대기질 측정소 위치는 한국환경공단에서 운영하는 에어코리아(Air Korea) 공식 홈페이지의 에어코리아 측정소 정보에서의 주소를 사용하여 위도, 경도 좌표를 생성함.
https://www.airkorea.or.kr/web/stationInfo?pMENU_NO=93

In [11]:
val urlStation = "./data/station_list.json"
val df_ObservatoryGyonggi = DataFrame.readJson(urlStation).update { colsOf<String>() }.with { it.trim() }
df_ObservatoryGyonggi.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
obs,String,111,111,0,가남읍,1,null,null,가남읍,동탄,송북동,의정부동,화도읍
addr,String,111,111,0,경기도 여주시 가남읍 태평중앙1길 20 가남읍행정복지센터 옥상,1,null,null,경기 고양시 덕양구 신원2로 24 신원도서관,경기 성남시 중원구 둔촌대로 425 상대원1동행정복지센터,경기 파주시 와석순환로 470 한국토지주택공사 파주사업본부,경기도 안산시 상록구 오목로7길 15 본오1동 작은도서관 (본오동),경기도 화성시 우정읍 쌍봉로 109-14 우정읍 행정복지센터
op,String,111,2,0,경기도보건환경연구원,110,null,null,경기도보건환경연구원,경기도보건환경연구원,경기도보건환경연구원,경기도보건환경연구원,한국환경공단 수도권동부환경본부
regdate,String,111,33,0,2020,14,null,null,1986,1999,2005,2019,2025
lng,Float,111,111,0,127.544876,1,127.039223,0.238667,126.552834,126.842580,127.040810,127.171160,127.629669
lat,Float,111,111,0,37.201645,1,37.423565,0.241476,36.974628,37.280584,37.380257,37.616242,38.096458


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
obs,String,111,111,0,가남읍,1,null,null,가남읍,동탄,송북동,의정부동,화도읍
addr,String,111,111,0,경기도 여주시 가남읍 태평중앙1길 20 가남읍행정복지센터 옥상,1,null,null,경기 고양시 덕양구 신원2로 24 신원도서관,경기 성남시 중원구 둔촌대로 425 상대원1동행정복지센터,경기 파주시 와석순환로 470 한국토지주택공사 파주사업본부,경기도 안산시 상록구 오목로7길 15 본오1동 작은도서관 (본오동),경기도 화성시 우정읍 쌍봉로 109-14 우정읍 행정복지센터
op,String,111,2,0,경기도보건환경연구원,110,null,null,경기도보건환경연구원,경기도보건환경연구원,경기도보건환경연구원,경기도보건환경연구원,한국환경공단 수도권동부환경본부
regdate,String,111,33,0,2020,14,null,null,1986,1999,2005,2019,2025
lng,Float,111,111,0,127.544876,1,127.039223,0.238667,126.552834,126.842580,127.040810,127.171160,127.629669
lat,Float,111,111,0,37.201645,1,37.423565,0.241476,36.974628,37.280584,37.380257,37.616242,38.096458


~~~
import java.sql.DriverManager
val tableName = "KDoT_Location"
DriverManager.getConnection(dbPath).use{ conn ->

    val sql = """INSERT INTO ${tableName} (obs, addr, op, regdate, lng, lat) VALUES (?,?,?,?,?,? )""".trimIndent()
    df.forEach { it ->
        try {
            conn.prepareStatement(sql)?.use { preparedStatement ->
                preparedStatement.setString(1, it["obs"].toString())
                preparedStatement.setString(2, it["addr"].toString())
                preparedStatement.setString(3, it["op"].toString())
                preparedStatement.setString(4, it["regdate"].toString())
                preparedStatement.setString(5, it["lng"].toString())
                preparedStatement.setString(6, it["lat"].toString())
                preparedStatement.executeUpdate()
            }
        } catch (e: Exception){
            println(e.localizedMessage)
        }
    }

}
~~~

In [ ]:
@file:OptIn(ExperimentalTime::class)

import kotlinx.datetime.format.FormatStringsInDatetimeFormats
import kotlinx.datetime.format.byUnicodePattern
import kotlin.time.ExperimentalTime

@OptIn(ExperimentalTime::class)
val now = Clock.System.now()

@OptIn(FormatStringsInDatetimeFormats::class, ExperimentalTime::class)
val previous1Hour = now
    .minus(2, DateTimeUnit.HOUR)
    .toLocalDateTime(TimeZone.of("Asia/Seoul"))
    .format(LocalDateTime.Format{byUnicodePattern("yyyy-MM-dd HH")})

println("Current time : ${now}, Previous time : ${previous1Hour}")

val endPoint = "https://openapi.gg.go.kr"
val service = "Sidoatmospolutnmesure"
val apiKey = ""
val type = "json"

val mesure_day_tm = "${previous1Hour.encodeURLParameter()}:00"

val url = "${endPoint}/${service}?KEY=${apiKey}&Type=${type}&MESURE_DAY_TM=${mesure_day_tm}"

In [23]:

fun loadData(path:String, maxPage:Int): List<DataFrame<*>> {
    val rows = mutableListOf<DataFrame<*>>()
    var requestPage = 1
    do{
        val pagePath = "$path&pIndex=$requestPage"
        val jsonData = DataFrame.readJson(pagePath)
        try {
            val instanceDf = (jsonData["Sidoatmospolutnmesure"][0] as DataFrame<*>)["row"][1] as DataFrame<*>
            requestPage += 1
            rows.add(instanceDf)
        } catch(e: Exception) {
            print(e.localizedMessage)
            break
        }
    } while (requestPage <= maxPage )
    return rows
}


In [24]:
val df_ObservationRawGyonggiValues = loadData(url, 2).concat().update { colsOf<String>() }.with { it?.trim() }
df_ObservationRawGyonggiValues.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
SIGUN_CD,String,126,31,0,41110,8,null,null,41110,41220,41370,41550,41830
SIGUN_NM,String,126,31,0,수원시,8,null,null,가평군,부천시,안산시,의왕시,화성시
MESURSTN_NM,String,126,126,0,소사본동,1,null,null,가남읍,동구동,소사본동,의정부동,화도읍
INSTL_YY,String,126,34,0,2020,17,null,null,1986,2000,2006,2019,2025
MESRNW_NM,String,126,3,0,도시대기,111,null,null,교외대기,도시대기,도시대기,도시대기,도시대기
MESURE_DAY_TM,String,126,1,0,2026-06-10 12:00,126,null,null,2026-06-10 12:00,2026-06-10 12:00,2026-06-10 12:00,2026-06-10 12:00,2026-06-10 12:00
SUA_GAS_DNST_VL,Float?,126,6,11,0.002000,47,0.002696,0.000975,0.001000,0.002000,0.003000,0.003000,0.005000
COMNXD_DNST_VL,Float?,126,5,9,0.300000,70,0.339316,0.065597,0.200000,0.300000,0.300000,0.400000,0.500000
NO2_DNST_VL,Float?,126,17,8,0.006000,22,0.007864,0.003186,0.002000,0.006000,0.007000,0.010000,0.019000
OZONE_DNST_VL,Float?,126,35,8,0.083000,8,0.079949,0.008353,0.054000,0.074000,0.080000,0.086000,0.100000


In [25]:
val df_ObservationRawGyonggiValuesRename = df_ObservationRawGyonggiValues.rename(
    "MESURSTN_NM" to "obs",
    "SUA_GAS_DNST_VL" to "SO2",
    "COMNXD_DNST_VL" to "CO",
    "NO2_DNST_VL" to "NO2",
    "OZONE_DNST_VL" to "O3",
    "FINEDUST_PM10_DNST_VL" to "PM10",
    "FINEDUST_PM2_5_DNST_VL" to "PM2_5"
).rename{ all() }.into{ it.name.lowercase() }
df_ObservationRawGyonggiValuesRename.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
sigun_cd,String,126,31,0,41110,8,null,null,41110,41220,41370,41550,41830
sigun_nm,String,126,31,0,수원시,8,null,null,가평군,부천시,안산시,의왕시,화성시
obs,String,126,126,0,소사본동,1,null,null,가남읍,동구동,소사본동,의정부동,화도읍
instl_yy,String,126,34,0,2020,17,null,null,1986,2000,2006,2019,2025
mesrnw_nm,String,126,3,0,도시대기,111,null,null,교외대기,도시대기,도시대기,도시대기,도시대기
mesure_day_tm,String,126,1,0,2026-06-10 12:00,126,null,null,2026-06-10 12:00,2026-06-10 12:00,2026-06-10 12:00,2026-06-10 12:00,2026-06-10 12:00
so2,Float?,126,6,11,0.002000,47,0.002696,0.000975,0.001000,0.002000,0.003000,0.003000,0.005000
co,Float?,126,5,9,0.300000,70,0.339316,0.065597,0.200000,0.300000,0.300000,0.400000,0.500000
no2,Float?,126,17,8,0.006000,22,0.007864,0.003186,0.002000,0.006000,0.007000,0.010000,0.019000
o3,Float?,126,35,8,0.083000,8,0.079949,0.008353,0.054000,0.074000,0.080000,0.086000,0.100000


In [26]:
val rightOnly = df_ObservationRawGyonggiValuesRename.excludeJoin(df_ObservatoryGyonggi){ obs }
rightOnly.sortBy { "sigun_nm" and "obs" }

sigun_cd,sigun_nm,obs,instl_yy,mesrnw_nm,mesure_day_tm,so2,co,no2,o3,pm10,pm2_5
41280,고양시,백마로(마두역),2004,도로변대기,2026-06-10 12:00,0.003000,0.300000,0.010000,0.071000,54.000000,31.000000
41570,김포시,한강로,2020,도로변대기,2026-06-10 12:00,0.003000,0.300000,0.007000,0.072000,46.000000,29.000000
41360,남양주시,경춘로,2019,도로변대기,2026-06-10 12:00,0.002000,0.300000,0.016000,0.060000,56.000000,32.000000
41190,부천시,송내대로(중동),2004,도로변대기,2026-06-10 12:00,0.002000,0.400000,0.011000,0.079000,61.000000,29.000000
41130,성남시,대왕판교로(백현동),2009,도로변대기,2026-06-10 12:00,0.002000,0.400000,0.018000,0.077000,38.000000,18.000000
41130,성남시,성남대로(모란역),2004,도로변대기,2026-06-10 12:00,0.002000,0.500000,0.018000,0.080000,37.000000,25.000000
41110,수원시,경수대로(동수원),2004,도로변대기,2026-06-10 12:00,0.002000,0.500000,0.019000,0.077000,38.000000,20.000000
41390,시흥시,서해안로,2020,도로변대기,2026-06-10 12:00,0.004000,0.300000,0.012000,0.076000,58.000000,33.000000
41270,안산시,중앙대로(고잔동),2009,도로변대기,2026-06-10 12:00,0.002000,0.300000,0.013000,0.071000,50.000000,13.000000
41800,연천군,연천(DMZ),2020,교외대기,2026-06-10 12:00,0.001000,0.200000,0.002000,0.059000,33.000000,18.000000


In [27]:
val df_dataGyonggi = df_ObservatoryGyonggi.leftJoin(df_ObservationRawGyonggiValuesRename){ obs }
df_dataGyonggi.sortBy { "sigun_nm" and "obs" }

obs,addr,op,regdate,lng,lat,sigun_cd,sigun_nm,instl_yy,mesrnw_nm,mesure_day_tm,so2,co,no2,o3,pm10,pm2_5
가평,경기도 가평군 가평읍 석봉로 181 가평군청 의회동,경기도보건환경연구원,2010,127.509705,37.831310,41820,가평군,2010,도시대기,2026-06-10 12:00,0.001000,0.300000,0.005000,0.069000,25.000000,20.000000
설악면,경기도 가평군 설악면 한서로 8 설악면 문화센터 옥상,경기도보건환경연구원,2020,127.494080,37.676113,41820,가평군,2020,도시대기,2026-06-10 12:00,0.001000,0.400000,0.005000,0.070000,25.000000,12.000000
식사동,경기 고양시 일산동구 위시티로 151 양일초등학교,경기도보건환경연구원,2002,126.813599,37.685337,41280,고양시,2002,도시대기,2026-06-10 12:00,0.002000,0.200000,0.004000,0.084000,42.000000,27.000000
신원동,경기 고양시 덕양구 신원2로 24 신원도서관,경기도보건환경연구원,2015,126.886353,37.666378,41280,고양시,2015,도시대기,2026-06-10 12:00,0.003000,0.300000,0.004000,0.086000,38.000000,26.000000
주엽동,경기도 고양시 일산서구 주엽로 104 주엽어린이도서관 (주엽동),경기도보건환경연구원,2018,126.756493,37.668461,41280,고양시,2018,도시대기,2026-06-10 12:00,0.003000,0.300000,0.004000,0.075000,53.000000,38.000000
행신동,경기 고양시 덕양구 화신로 148 행신배수지 가라산공원,경기도보건환경연구원,1998,126.841988,37.625172,41280,고양시,1998,도시대기,2026-06-10 12:00,null,null,null,null,null,null
과천동,경기 과천시 상하벌로 17 과천시 환경사업소,경기도보건환경연구원,2000,127.000771,37.448727,41290,과천시,2000,도시대기,2026-06-10 12:00,0.002000,0.500000,0.007000,0.100000,45.000000,19.000000
별양동,경기 과천시 코오롱로 53 문원초등학교,경기도보건환경연구원,1991,126.995003,37.424015,41290,과천시,1991,도시대기,2026-06-10 12:00,0.001000,0.400000,0.006000,0.087000,35.000000,24.000000
소하동,경기 광명시 소하일로 7 소하도서관,경기도보건환경연구원,1998,126.887993,37.445457,41210,광명시,1998,도시대기,2026-06-10 12:00,0.002000,0.400000,0.005000,0.096000,45.000000,18.000000
철산동,경기 광명시 시청로 20 광명시청 제1별관,경기도보건환경연구원,1986,126.865227,37.478569,41210,광명시,1986,도시대기,2026-06-10 12:00,0.002000,0.400000,0.010000,0.081000,39.000000,20.000000
